# 6️⃣ Flood Detection — DeepLabV3+ (The Context King) 👑

**Why this is the ultimate single-model architecture:**
Unlike UNet++, DeepLabV3+ uses ASPP (Atrous Spatial Pyramid Pooling). Instead of focusing on local pixel borders, it looks at the image through a massive "wide-angle lens" simultaneously. It intuitively understands that a flood is part of a larger geographic structure rather than just a puddle.

**Features included:**
1. `efficientnet-b5` Heavy Backbone
2. MNDWI Band Engineering (7th Channel)
3. Modal Dropout (Cloud Simulation)
4. Automated Threshold Tuning to maximize Flood IoU
5. Full Test-Time Augmentation (TTA)




## Cell 1 — Install & Imports




In [ ]:
!pip install -q segmentation-models-pytorch albumentations rasterio

import os
import gc
import random
import numpy as np
import pandas as pd
from pathlib import Path
from glob import glob

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2
import rasterio

gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()



## Cell 2 — Config & Paths




In [ ]:
DATA_ROOT = '/kaggle/input/anrfaisehack-theme-1-phase2/data/'
OUT_DIR = '/kaggle/working/'

EPOCHS = 60
BATCH_SIZE = 6 # Dropped to 6 because DeepLabV3+ with efficientnet-b5 is heavier
LR = 1.0e-3

MEANS = [841.1, 371.6, 1734.1, 1588.3, 1742.8, 1218.5]
STDS = [473.7, 170.3, 623.0, 612.8, 564.5, 528.0]

pl_seed = 42
torch.manual_seed(pl_seed)
torch.cuda.manual_seed(pl_seed)
np.random.seed(pl_seed)
random.seed(pl_seed)



## Cell 3 — Dataset with MNDWI & Modal Dropout




In [ ]:
class FloodDataset(Dataset):
    def __init__(self, split_file, transform=None, is_train=False):
        self.transform = transform
        self.is_train = is_train
        
        with open(split_file) as f: ids = [l.strip() for l in f if l.strip()]
        self.imgs = [os.path.join(DATA_ROOT + 'image/', f"{i}_image.tif") for i in ids]
        self.lbls = [os.path.join(DATA_ROOT + 'label/', f"{i}_label.tif") for i in ids]

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        with rasterio.open(self.imgs[idx]) as src:
            raw = src.read().astype(np.float32)
            
        # 🌟 MNDWI Band Engineering
        mndwi = (raw[2] - raw[5]) / (raw[2] + raw[5] + 1e-8)
        
        img = raw.copy()
        for b in range(6): img[b] = (img[b] - MEANS[b]) / STDS[b]
        
        img = np.nan_to_num(img, nan=0.0)
        mndwi = np.nan_to_num(mndwi, nan=0.0)
        img = np.concatenate([img, np.expand_dims(mndwi, axis=0)], axis=0)

        # 🌟 Modal Dropout (Train only)
        if self.is_train:
            r = random.random()
            if r < 0.20: img[2:6, :, :] = 0.0  # Drop optical
            elif r < 0.30: img[0:2, :, :] = 0.0 # Drop SAR

        img = img.transpose(1, 2, 0)
        with rasterio.open(self.lbls[idx]) as src: mask = src.read(1).astype(np.int64)

        if self.transform:
            aug = self.transform(image=img, mask=mask)
            img, mask = aug['image'], aug['mask']
        else:
            img, mask = torch.from_numpy(img.transpose(2,0,1)), torch.from_numpy(mask)

        return img, mask

t_trans = A.Compose([A.D4(), A.RandomCrop(512, 512), ToTensorV2()])
v_trans = A.Compose([ToTensorV2()])

train_ds = FloodDataset(DATA_ROOT + 'split/train.txt', t_trans, is_train=True)
val_ds   = FloodDataset(DATA_ROOT + 'split/val.txt', v_trans)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)



## Cell 4 — DeepLabV3+ & Lovasz Loss




In [ ]:
# 🌟 Upgraded Architecture
model = smp.DeepLabV3Plus(
    encoder_name='efficientnet-b5', 
    encoder_weights="imagenet", 
    in_channels=7, 
    classes=3
).cuda()

class KaggleWinnerLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.lovasz = smp.losses.LovaszLoss('multiclass', ignore_index=-1)
        self.focal = smp.losses.FocalLoss('multiclass', ignore_index=-1, gamma=2.0)
    def forward(self, logits, masks):
        return 0.7 * self.lovasz(logits, masks) + 0.3 * self.focal(logits, masks)

criterion = KaggleWinnerLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler('cuda')



## Cell 5 — Execute Training! 🚀




In [ ]:
best_model_path = os.path.join(OUT_DIR, "best_deeplabv3.pth")

# Custom IoU metric for pure numpy arrays
def calc_flood_iou(pred, target):
    p_flood = (pred == 1)
    t_flood = (target == 1)
    inter = (p_flood & t_flood).sum()
    union = (p_flood | t_flood).sum()
    return inter / union if union > 0 else 0.0

best_val_iou = 0.0

for epoch in range(EPOCHS):
    model.train()
    for imgs, masks in train_loader:
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            loss = criterion(model(imgs.cuda()), masks.cuda())
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    scheduler.step()
    
    # Eval Loop for Flood IoU
    model.eval()
    val_ious = []
    with torch.no_grad(), torch.amp.autocast('cuda'):
        for imgs, masks in val_loader:
            preds = model(imgs.cuda()).argmax(dim=1).cpu().numpy()
            for i in range(preds.shape[0]):
                val_ious.append(calc_flood_iou(preds[i], masks[i].numpy()))
                
    mean_flood_iou = np.mean(val_ious)
    
    if mean_flood_iou > best_val_iou:
        best_val_iou = mean_flood_iou
        torch.save(model.state_dict(), best_model_path)
        
    if epoch % 5 == 0 or epoch == EPOCHS - 1:
        print(f"Epoch {epoch:02d} | Val Flood IoU: {mean_flood_iou:.4f} " + ("★" if mean_flood_iou == best_val_iou else ""))

print(f"✅ Training Done. Best Validation Flood IoU: {best_val_iou:.4f}")



## Cell 6 — Automated Threshold Sweep




In [ ]:
model.load_state_dict(torch.load(best_model_path, weights_only=True))
model.eval()

print("🔍 Sweeping Validation Set for absolute perfect Flood Threshold...")
thresholds = np.arange(0.20, 0.70, 0.05)
best_thresh = 0.50
best_sweep_iou = 0.0

val_metrics = {t: [] for t in thresholds}

with torch.no_grad(), torch.amp.autocast('cuda'):
    for imgs, masks in val_loader:
        logits = model(imgs.cuda())
        probs = F.softmax(logits, dim=1).cpu().numpy()
        masks = masks.numpy()
        
        base_preds = np.argmax(probs[:, [0, 2], :, :], axis=1) 
        base_preds[base_preds == 1] = 2 
        
        for t in thresholds:
            preds = base_preds.copy()
            preds[probs[:, 1, :, :] > t] = 1 
            for i in range(preds.shape[0]):
                val_metrics[t].append(calc_flood_iou(preds[i], masks[i]))

print("--- Sweep Results ---")
for t in thresholds:
    mean_iou = np.mean(val_metrics[t])
    print(f"Threshold {t:.2f}: {mean_iou:.4f}")
    if mean_iou > best_sweep_iou:
        best_sweep_iou = mean_iou
        best_thresh = t

print(f"✅ LOCKED OPTIMAL THRESHOLD: {best_thresh:.2f}")



## Cell 7 — TTA Predict & Generate Submission




In [ ]:
def mask_to_rle(mask):
    pixels = mask.flatten(order="F")
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(str(x) for x in runs)

predict_files = sorted(Path(DATA_ROOT + 'prediction/image/').glob("*.tif"))
rows = []

for p in predict_files:
    with rasterio.open(p) as src: raw = src.read().astype(np.float32)
        
    mndwi = (raw[2] - raw[5]) / (raw[2] + raw[5] + 1e-8)
    for b in range(6): raw[b] = (raw[b] - MEANS[b]) / STDS[b]
    raw = np.nan_to_num(raw, nan=0.0)
    img = np.concatenate([raw, np.expand_dims(mndwi, axis=0)], axis=0) 
    
    x = torch.from_numpy(img).unsqueeze(0).float().cuda()
    with torch.no_grad(), torch.amp.autocast('cuda'):
        # Triple TTA (Original, V-Flip, H-Flip)
        p1 = F.softmax(model(x), dim=1)
        p2 = torch.flip(F.softmax(model(torch.flip(x, dims=[2])), dim=1), dims=[2])
        p3 = torch.flip(F.softmax(model(torch.flip(x, dims=[3])), dim=1), dims=[3])
        probs = ((p1 + p2 + p3) / 3.0).squeeze(0).cpu().numpy()
        
    # Apply Swept Threshold dynamically!
    base_calc = np.argmax(probs[[0, 2], :, :], axis=0) 
    pred = np.where(base_calc == 1, 2, 0)              
    pred[probs[1, :, :] > best_thresh] = 1             
    
    flood_mask = (pred == 1).astype(np.uint8)
    rle = mask_to_rle(flood_mask)
    rows.append({"id": p.name.replace("_image.tif", ""), "rle_mask": rle if rle.strip() else "0 0"})

df = pd.DataFrame(rows)
df.to_csv("submission_master_deeplabv3.csv", index=False)
print(f"🎉 MASTER SCRIPT FINISHED! Generated submission_master_deeplabv3.csv")
